In [1]:
# ############################################################
# Level 3 — 타이타닉 생존 예측 + 계수 해석 (공개 데이터)
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 불러오기 — 표 / 분리 / 전처리 / 모델 / 평가
# ------------------------------------------------------------
import pandas as pd                               # 표(CSV) 다루기
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# ------------------------------------------------------------
# [목적] 데이터 불러오기 + 전처리 (결측 채우기 / 글자 -> 0·1)
# ------------------------------------------------------------
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)                              # 공개 URL에서 바로
df['Age'] = df['Age'].fillna(df['Age'].median())  # 나이 결측을 중앙값으로
X = pd.get_dummies(df[['Pclass','Sex','Age','Fare']], drop_first=True)  # 성별 -> 0/1 (중복 방지)
y = df['Survived']                                # 정답 (1=생존)

In [3]:
# ------------------------------------------------------------
# [목적] 분리 -> 스케일링 -> 학습 -> 정확도 + 계수
# ------------------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)   # 비율 유지 8:2
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)                   # train만 기준 학습
X_te_s = sc.transform(X_te)                       # test는 변환만
model = LogisticRegression(max_iter=5000).fit(X_tr_s, y_tr)   # 학습

print('정확도:', round(accuracy_score(y_te, model.predict(X_te_s)), 3))
print('계수(양수=생존확률 UP):'); print(pd.Series(model.coef_[0], index=X.columns).round(2))

정확도: 0.799
계수(양수=생존확률 UP):
Pclass     -0.97
Age        -0.43
Fare        0.08
Sex_male   -1.25
dtype: float64


In [4]:
# ============================================================
# [결과 해석]
#  · 정확도 ~ 0.80 -> 생존 여부를 꽤 잘 맞힘
#  · 계수 읽기:
#      - Sex_male ~ -1.25 -> '남성'일수록 생존확률 크게 하락 (여성·아이 우선 구조)
#      - Pclass  ~ -0.97 -> 등급 숫자가 클수록(3등석) 생존확률 하락
#      - Age     ~ -0.43 -> 나이 많을수록 소폭 하락
#      - Fare    ~ +0.08 -> 요금 높을수록 소폭 상승
#  · 실제 역사(여성·상류층 생존율↑)와 방향이 일치 -> 계수로 '이유'를 설명 가능
# ============================================================